extract valid and needed matches

In [ ]:
import pandas as pd

PATH = r"data_processed/match_master_clean_mergedwithclimate_teamnormalized.csv"


df = pd.read_csv(PATH, usecols=["match_id"])


match_ids = df["match_id"].dropna().unique()

print(f"Total unique match_ids: {len(match_ids)}")
print(match_ids[:10])  


Total unique match_ids: 6055
[636164 636165 636166 636536 636537 636538 648681 648683 652257 652259]


In [2]:
pd.DataFrame({"match_id": match_ids}).to_csv(
    "data_processed/filtered_match_ids.csv",
    index=False
)


converting match ids into sets

In [6]:
match_id_set = set(match_ids)
print(match_id_set)


{np.int64(851969), np.int64(851971), np.int64(851973), np.int64(851975), np.int64(851977), np.int64(851979), np.int64(851981), np.int64(851983), np.int64(851985), np.int64(851989), np.int64(1277974), np.int64(851991), np.int64(1277975), np.int64(1277976), np.int64(754039), np.int64(1409474), np.int64(1409475), np.int64(1409476), np.int64(1343941), np.int64(1343942), np.int64(1343943), np.int64(1343944), np.int64(1343945), np.int64(1343946), np.int64(1343947), np.int64(1343948), np.int64(1343949), np.int64(1343950), np.int64(1343951), np.int64(1343952), np.int64(1343953), np.int64(1343954), np.int64(1343955), np.int64(1343956), np.int64(1343957), np.int64(1343958), np.int64(1343959), np.int64(1343960), np.int64(1343961), np.int64(1343962), np.int64(1343963), np.int64(1343964), np.int64(1343965), np.int64(1343966), np.int64(1343967), np.int64(1343968), np.int64(1343969), np.int64(1343970), np.int64(1343971), np.int64(1343972), np.int64(1343973), np.int64(1409501), np.int64(1409502), np.i

extracting ball by ball data

In [ ]:
import os
import json
import pandas as pd

RAW_DIR = "RawData/all"
MATCH_ID_CSV = "data_processed/filtered_match_ids.csv"

OUT_BALL_CSV = "data_processed/ball_by_ball_clean.csv"
OUT_MISSING_CSV = "data_processed/missing_match_ids.csv"

TOTAL_LEGAL_BALLS = 120  # T20 = 120 legal balls


match_id_df = pd.read_csv(MATCH_ID_CSV, dtype=str)

match_id_set = set(
    match_id_df["match_id"]
    .astype(str)
    .str.strip()
)

print("Loaded match IDs:", len(match_id_set))

rows = []
found_match_ids = set()


for file in os.listdir(RAW_DIR):
    if not file.endswith(".json"):
        continue

    match_id = file.replace(".json", "").strip()

    # FILTER BY MATCH IDS
    if match_id not in match_id_set:
        continue

    found_match_ids.add(match_id)

    with open(os.path.join(RAW_DIR, file), "r", encoding="utf-8") as f:
        match = json.load(f)

    info = match.get("info", {})
    teams = info.get("teams", [])

  
    if len(teams) != 2:
        continue

    team_a, team_b = teams

    innings_data = match.get("innings", [])
    first_innings_total = None

    for inn_idx, inn in enumerate(innings_data, start=1):

        batting_team = inn.get("team")
        bowling_team = team_b if batting_team == team_a else team_a

        overs = inn.get("overs", [])

        runs_so_far = 0
        wickets = 0
        legal_balls = 0

        # TARGET (2ND INNINGS ONLY)
        target = None
        if inn_idx == 2 and first_innings_total is not None:
            target = first_innings_total + 1

        for over_data in overs:
            over_no = over_data.get("over")

            for ball_no, d in enumerate(over_data.get("deliveries", []), start=1):

                runs = d.get("runs", {})
                bat_runs = runs.get("batter", 0)
                extra_runs = runs.get("extras", 0)
                total_runs = runs.get("total", 0)

                # WICKET INFO 
                is_wicket = 1 if d.get("wickets") else 0
                fielder = None
                if d.get("wickets"):
                    fielders = d["wickets"][0].get("fielders")
                    if fielders:
                        fielder = fielders[0].get("name")

                #LEGAL BALL CHECK
                extras_type = d.get("extras", {})
                is_legal = not ("wides" in extras_type or "noballs" in extras_type)

                if is_legal:
                    legal_balls += 1

                #UPDATE STATE
                runs_so_far += total_runs
                wickets += is_wicket

                balls_remaining = max(0, TOTAL_LEGAL_BALLS - legal_balls)

                #RATE METRICS
                current_rr = (
                    runs_so_far / (legal_balls / 6)
                    if legal_balls > 0 else None
                )

                runs_remaining = None
                required_rr = None
                req_per_ball = None

                if target is not None and balls_remaining > 0:
                    runs_remaining = target - runs_so_far
                    required_rr = runs_remaining / (balls_remaining / 6)
                    req_per_ball = runs_remaining / balls_remaining

                rows.append({

                    "match_id": match_id,
                    "innings": inn_idx,
                    "over": over_no,
                    "ball": ball_no,


                    "batting_team": batting_team,
                    "bowling_team": bowling_team,


                    "batter": d.get("batter"),
                    "bowler": d.get("bowler"),
                    "fielder": fielder,


                    "runs_off_bat": bat_runs,
                    "extras": extra_runs,
                    "total_runs": total_runs,
                    "is_wicket": is_wicket,


                    "runs_remaining": runs_remaining,
                    "balls_remaining": balls_remaining,
                    "wickets_lost": wickets,
                    "current_run_rate": current_rr,
                    "required_run_rate": required_rr,
                    "req_per_ball": req_per_ball,
                    "rr_diff": (
                        current_rr - required_rr
                        if current_rr is not None and required_rr is not None
                        else None
                    ),
                    "is_second_innings": 1 if inn_idx == 2 else 0,
                    "phase_powerplay": 1 if over_no < 6 else 0,
                    "phase_death": 1 if balls_remaining <= 30 else 0
                })

        #STORE FIRST INNINGS TOTAL
        if inn_idx == 1:
            first_innings_total = runs_so_far


bb_df = pd.DataFrame(rows)
bb_df.to_csv(OUT_BALL_CSV, index=False)


missing_ids = sorted(match_id_set - found_match_ids)
missing_df = pd.DataFrame({"match_id": missing_ids})
missing_df.to_csv(OUT_MISSING_CSV, index=False)


print(" Ball-by-ball extraction complete")
print("Saved:", OUT_BALL_CSV)
print("Total rows:", len(bb_df))
print(" Missing JSON files:", len(missing_ids))
if missing_ids:
    print("Saved:", OUT_MISSING_CSV)


Loaded match IDs: 6055
✅ Ball-by-ball extraction complete
Saved: data_processed/ball_by_ball_clean.csv
Total rows: 1394613
⚠️ Missing JSON files: 0


adding match and climate context

In [ ]:
import pandas as pd


BALL_PATH = "data_processed/ball_by_ball_clean.csv"
MATCH_MASTER_PATH =  "data_processed/match_master_clean_mergedwithclimate_teamnormalized.csv"

OUT_PATH = "data_processed/ball_by_ball_with_match_context.csv"
OUT_UNMATCHED = "data_processed/unmatched_ball_rows.csv"


MATCH_JOIN_COLS = [
    "match_id",
    "date",
    "league",
    "team1",
    "team2",
    "venue",
    "city",
    "latitude",
    "longitude",
    "tavg",
    "tmax",
    "tmin",
    "rhum",
    "prcp",
    "wspd",
    "pres",
    "tsun"
]


print("Loading ball-by-ball data...")
ball_df = pd.read_csv(
    BALL_PATH,
    dtype={"match_id": str},
    low_memory=False
)

print("Ball rows:", len(ball_df))

print("Loading match master...")
match_df = pd.read_csv(
    MATCH_MASTER_PATH,
    usecols=MATCH_JOIN_COLS,
    dtype={"match_id": str}
)

print("Match rows:", len(match_df))


dup_count = match_df["match_id"].duplicated().sum()
if dup_count > 0:
    print(f" Warning: {dup_count} duplicate match_id rows found in match master")
    match_df = match_df.drop_duplicates("match_id")


print("Merging...")
merged_df = ball_df.merge(
    match_df,
    on="match_id",
    how="left",
    validate="many_to_one"
)


print("\n--- SANITY CHECKS ---")


print("Rows before merge :", len(ball_df))
print("Rows after merge  :", len(merged_df))

assert len(ball_df) == len(merged_df), " Row count changed after merge!"


missing_mask = merged_df["venue"].isna()
missing_count = missing_mask.sum()

print("Rows missing match context:", missing_count)

if missing_count > 0:
    print("⚠️ Saving unmatched rows for inspection")
    merged_df[missing_mask].to_csv(OUT_UNMATCHED, index=False)
    print("Saved:", OUT_UNMATCHED)


required_cols = ["venue", "city", "league"]
for col in required_cols:
    null_pct = merged_df[col].isna().mean()
    print(f"{col} null %:", round(null_pct * 100, 3))


merged_df.to_csv(OUT_PATH, index=False)

print("\n MERGE COMPLETE")
print("Saved:", OUT_PATH)
print("Final row count:", len(merged_df))


Loading ball-by-ball data...
Ball rows: 1394613
Loading match master...
Match rows: 6055
Merging...

--- SANITY CHECKS ---
Rows before merge : 1394613
Rows after merge  : 1394613
Rows missing match context: 0
venue null %: 0.0
city null %: 11.812
league null %: 0.0

✅ MERGE COMPLETE
Saved: data_processed/ball_by_ball_with_match_context.csv
Final row count: 1394613


sorting the data

In [ ]:
import pandas as pd

PATH = "data_processed/ball_by_ball_with_match_context.csv"

df = pd.read_csv(
    PATH,
    parse_dates=["date"],
    low_memory=False
)

# Sort chronologically
df = df.sort_values(
    by=["date", "match_id", "innings", "over", "ball"]
).reset_index(drop=True)

# Save back
df.to_csv(PATH, index=False)

print(" Sorted ball-by-ball data by date")


✅ Sorted ball-by-ball data by date


normalising names for ball by ball

In [ ]:
import pandas as pd


IN_CSV  = "data_processed/ball_by_ball_with_match_context.csv"
OUT_CSV = "data_processed/ball_by_ball_with_match_context_teamnormalized.csv"


df = pd.read_csv(IN_CSV, low_memory=False)


# TEAM RENAME MAP

TEAM_RENAME_MAP = {
    # IPL
    "royal challengers bangalore": "royal challengers bengaluru",
    "royal challengers bengaluru": "royal challengers bengaluru",
    "delhi daredevils": "delhi capitals",
    "delhi capitals": "delhi capitals",
    "kings xi punjab": "punjab kings",
    "punjab kings": "punjab kings",
    "deccan chargers": "sunrisers hyderabad",
    "sunrisers hyderabad": "sunrisers hyderabad",
    "rising pune supergiant": "rising pune supergiant",
    "rising pune supergiants": "rising pune supergiant",

    # BPL
    "chittagong kings": "chattogram royals",
    "chittagong vikings": "chattogram royals",
    "chattogram challengers": "chattogram royals",
    "chattogram royals": "chattogram royals",
    "dhaka gladiators": "dhaka capitals",
    "dhaka dynamites": "dhaka capitals",
    "dhaka platoon": "dhaka capitals",
    "minister dhaka": "dhaka capitals",
    "dhaka dominators": "dhaka capitals",
    "durdanto dhaka": "dhaka capitals",
    "dhaka capitals": "dhaka capitals",
    "sylhet royals": "sylhet titans",
    "sylhet super stars": "sylhet titans",
    "sylhet sixers": "sylhet titans",
    "sylhet thunder": "sylhet titans",
    "sylhet sunrisers": "sylhet titans",
    "sylhet strikers": "sylhet titans",
    "sylhet titans": "sylhet titans",
    "barisal burners": "fortune barishal",
    "barisal bulls": "fortune barishal",
    "fortune barishal": "fortune barishal",
    "duronto rajshahi": "rajshahi warriors",
    "rajshahi kings": "rajshahi warriors",
    "rajshahi royals": "rajshahi warriors",
    "durbar rajshahi": "rajshahi warriors",
    "rajshahi warriors": "rajshahi warriors",
    "comilla victorians": "comilla victorians",
    "cumilla warriors": "comilla victorians",

    # CPL
    "barbados tridents": "barbados royals",
    "barbados royals": "barbados royals",
    "trinidad and tobago red steel": "trinbago knight riders",
    "trinbago knight riders": "trinbago knight riders",
    "st lucia zouks": "st. lucia kings",
    "st lucia stars": "st. lucia kings",
    "st. lucia kings": "st. lucia kings",

    # The Hundred
    "oval invincibles": "mi london",
    "mi london": "mi london",
    "manchester originals": "manchester super giants",
    "manchester super giants": "manchester super giants",

    # LPL
    "colombo kings": "colombo strikers",
    "colombo stars": "colombo strikers",
    "colombo strikers": "colombo strikers",
    "dambulla giants": "dambulla aura",
    "dambulla aura": "dambulla aura",
    "galle gladiators": "galle titans",
    "galle titans": "galle titans",
    "jaffna stallions": "jaffna kings",
    "jaffna kings": "jaffna kings",
    "kandy tuskers": "b-love kandy",
    "kandy falcons": "b-love kandy",
    "b-love kandy": "b-love kandy",
}


def normalize_team(name):
    if pd.isna(name):
        return name
    key = str(name).strip().lower()
    return TEAM_RENAME_MAP.get(key, key)


TEAM_COLUMNS = [
    "batting_team",
    "bowling_team",
    "team1",
    "team2",
    "winner"
]

for col in TEAM_COLUMNS:
    if col in df.columns:
        df[col] = df[col].apply(normalize_team)


df.to_csv(OUT_CSV, index=False)


print(" Saved normalized ball-by-ball file:")
print(OUT_CSV)

if "batting_team" in df.columns:
    print("\nSample batting teams:")
    print(sorted(df["batting_team"].dropna().unique())[:20])


✅ Saved normalized ball-by-ball file:
data_processed/ball_by_ball_with_match_context_teamnormalized.csv

Sample batting teams:
['abu dhabi knight riders', 'adelaide strikers', 'andhra', 'antigua and barbuda falcons', 'antigua hawksbills', 'arunachal pradesh', 'assam', 'auckland', 'australia', 'b-love kandy', 'bangladesh', 'barbados royals', 'baroda', 'bengal', 'bihar', 'birmingham bears', 'birmingham phoenix', 'boland', 'brisbane heat', 'canterbury']


capitalizing the team name

In [ ]:
import pandas as pd


IN_CSV  = "data_processed/ball_by_ball_with_match_context_teamnormalized.csv"
OUT_CSV = "data_processed/ball_by_ball_with_match_context_teamnormalized.csv"


df = pd.read_csv(IN_CSV, low_memory=False)


def title_case(name):
    if pd.isna(name):
        return name
    return name.strip().title()


TEAM_COLUMNS = [
    "batting_team",
    "bowling_team",
    "team1",
    "team2",
    "winner"
]

for col in TEAM_COLUMNS:
    if col in df.columns:
        df[col] = df[col].apply(title_case)


df.to_csv(OUT_CSV, index=False)


print(" Saved capitalized ball-by-ball file:", OUT_CSV)

if "batting_team" in df.columns:
    print("\nSample teams:")
    print(df[["batting_team", "bowling_team"]].head())


✅ Saved capitalized ball-by-ball file: data_processed/ball_by_ball_with_match_context_teamnormalized.csv

Sample teams:
          batting_team       bowling_team
0  Melbourne Renegades  Hobart Hurricanes
1  Melbourne Renegades  Hobart Hurricanes
2  Melbourne Renegades  Hobart Hurricanes
3  Melbourne Renegades  Hobart Hurricanes
4  Melbourne Renegades  Hobart Hurricanes


adding elo data 

In [ ]:
import pandas as pd


BALL_PATH = "data_processed/ball_by_ball_with_match_context_teamnormalized.csv"
ELO_PATH  = "data_processed/team_elo_history.csv"

OUT_CANONICAL = "data_processed/ball_by_ball_canonical.csv"
OUT_MISSING_ELO = "data_processed/matches_missing_elo.csv"


ball_df = pd.read_csv(
    BALL_PATH,
    dtype={"match_id": str},
    parse_dates=["date"],
    low_memory=False
)


elo_df = pd.read_csv(
    ELO_PATH,
    dtype={"match_id": str}
)

elo_df = elo_df[[
    "match_id",
    "team",
    "effective_elo",
    "matches_played"
]].rename(columns={
    "team": "team_name",
    "effective_elo": "elo"
})


# JOIN BATTING ELO

ball_df = ball_df.merge(
    elo_df.rename(columns={
        "team_name": "batting_team",
        "elo": "batting_elo",
        "matches_played": "batting_team_matches"
    }),
    on=["match_id", "batting_team"],
    how="left",
    validate="many_to_one"
)


# JOIN BOWLING ELO

ball_df = ball_df.merge(
    elo_df.rename(columns={
        "team_name": "bowling_team",
        "elo": "bowling_elo",
        "matches_played": "bowling_team_matches"
    }),
    on=["match_id", "bowling_team"],
    how="left",
    validate="many_to_one"
)

# ELO DIFF

ball_df["elo_diff"] = ball_df["batting_elo"] - ball_df["bowling_elo"]


# BAT FIRST FLAG

first_bat = (
    ball_df[ball_df["innings"] == 1]
    .groupby("match_id")["batting_team"]
    .first()
    .rename("bat_first_team")
)

ball_df = ball_df.merge(
    first_bat,
    on="match_id",
    how="left"
)

ball_df["bat_first_flag"] = (
    ball_df["batting_team"] == ball_df["bat_first_team"]
).astype(int)


ball_df["has_elo"] = ~(
    ball_df["batting_elo"].isna() |
    ball_df["bowling_elo"].isna()
)


missing_elo_matches = (
    ball_df.loc[~ball_df["has_elo"], "match_id"]
    .unique()
)

pd.DataFrame({"match_id": missing_elo_matches}).to_csv(
    OUT_MISSING_ELO, index=False
)


ball_df.to_csv(OUT_CANONICAL, index=False)

print(" Canonical dataset saved:", OUT_CANONICAL)
print("Matches missing ELO:", len(missing_elo_matches))
print("You can filter later using has_elo == True")


✅ Canonical dataset saved: data_processed/ball_by_ball_canonical.csv
Matches missing ELO: 213
You can filter later using has_elo == True


In [ ]:
import pandas as pd


MATCH_MASTER_PATH = "data_processed/match_master_clean_mergedwithclimate_teamnormalized.csv"


ELO_HISTORY_PATH = "data_processed/team_elo_history.csv"


match_df = pd.read_csv(
    MATCH_MASTER_PATH,
    dtype={"match_id": str}
)

elo_df = pd.read_csv(
    ELO_HISTORY_PATH,
    dtype={"match_id": str}
)


match_ids_match = set(match_df["match_id"].astype(str).str.strip())
match_ids_elo   = set(elo_df["match_id"].astype(str).str.strip())


common_ids = match_ids_match & match_ids_elo
missing_in_elo = match_ids_match - match_ids_elo
missing_in_match = match_ids_elo - match_ids_match


print("Total match_master match_ids :", len(match_ids_match))
print("Total elo_history match_ids  :", len(match_ids_elo))
print("Common match_ids             :", len(common_ids))
print("Missing in ELO               :", len(missing_in_elo))
print("Missing in match_master      :", len(missing_in_match))


print("\nSample missing in ELO (up to 10):")
print(list(missing_in_elo)[:10])

print("\nSample missing in match_master (up to 10):")
print(list(missing_in_match)[:10])


Total match_master match_ids : 6055
Total elo_history match_ids  : 5856
Common match_ids             : 5856
Missing in ELO               : 199
Missing in match_master      : 0

Sample missing in ELO (up to 10):
['1207789', '1250313', '730293', '1127491', '1369561', '1409487', '1068318', '857705', '652293', '1207713']

Sample missing in match_master (up to 10):
[]


In [25]:
import pandas as pd

def count_rows_without_elo(path="data_processed/ball_by_ball_canonical.csv"):
    """
    Counts rows where has_elo == False in the canonical ball-by-ball dataset.
    """
    df = pd.read_csv(path, usecols=["has_elo"])
    
    count = (~df["has_elo"]).sum()
    total = len(df)

    print(f"Rows without ELO : {count}")
    print(f"Total rows      : {total}")
    print(f"Percentage      : {100 * count / total:.2f}%")

    return count

missing_elo_rows = count_rows_without_elo()


Rows without ELO : 30787
Total rows      : 1394613
Percentage      : 2.21%
